# 03 — Gold AGREGADA para BI (Power BI · Kelly)

**Contrato §13.3 (datos del dashboard).** Materializa **tablas agregadas pequeñas** (no las 23M filas)
que responden la Pregunta de Oro vía las **dos palancas** del EDA. Todo sobre **base limpia**
(cuarentena 14–17 nov), salvo `agg_metricas_diarias` que conserva todos los días con banderas para
contar la calidad de datos.

> **Salida:** 6 CSV en un folder del Volume (`bi_export`). Descárgalos y **commitéalos en `reports/data/`**
> (excepción del `.gitignore`, §12.2); Power BI los consume desde ahí.
>
> **Consistencia:** se replican las definiciones del EDA (tabla UNIT, funnel, prize, segmentos) → los
> números coinciden con `eda_ecommerce.ipynb`. Si Kelly necesita un corte que no está, se añade una tabla más.

In [0]:
from pyspark.sql import functions as F
import pandas as pd, os

# 1. Definir Rutas
SILVER = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
GOLD   = "/Volumes/workspace/default/e_commerce/gold/features_session"
BI_OUT_SQL = "/Volumes/workspace/default/e_commerce/gold/bi_export_SQL"   # Nueva ruta para Spark SQL
os.makedirs(BI_OUT_SQL, exist_ok=True)

# 2. Cargar bases de datos completas
QUARANTINE = ["2019-11-14", "2019-11-15", "2019-11-16", "2019-11-17"]
silver_full = spark.read.format("delta").load(SILVER)
gold_full   = spark.read.format("delta").load(GOLD)

# Filtrar base limpia (negocio)
silver = silver_full.filter(~F.col("date").isin(QUARANTINE))
gold   = gold_full.filter(F.col("label_window_corrupt") == 0)
MISSING = "Unknown"

# 3. Materializar tabla UNIT temporal
# Mantenemos la lógica original por rendimiento de cluster Serverless
BI_UNITS = "/Volumes/workspace/default/e_commerce/gold/_tmp_bi_units"
(silver.groupBy("user_session", "product_id").agg(
    F.max(F.when(F.col("event_type") == "cart", 1).otherwise(0)).cast("boolean").alias("has_cart"),
    F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).cast("boolean").alias("has_purchase"),
    F.expr("percentile_approx(price, 0.5)").alias("price"),
    F.first("macro_category", ignorenulls=True).alias("macro_category"),
    F.first("brand", ignorenulls=True).alias("brand"))
 .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(BI_UNITS))

units = spark.read.format("delta").load(BI_UNITS)

# 4. REGISTRAR LAS VISTAS PARA EL MOTOR SPARK SQL
silver_full.createOrReplaceTempView("vw_silver_full")
silver.createOrReplaceTempView("vw_silver")
units.createOrReplaceTempView("vw_units")

# 5. Función de guardado ajustada a la nueva ruta
def to_csv_sql(pdf, name):
    pdf.to_csv(f"{BI_OUT_SQL}/{name}.csv", index=True)
    print(f"  -> {name}.csv  ({pdf.shape[0]} filas x {pdf.shape[1]} cols)")
    return pdf

print("✅ Bases limpias y Vistas SQL listas | UNIT:", units.count(), "| salida:", BI_OUT_SQL)

## 1. `agg_funnel_categoria` — ¿dónde se concentra la fuga? (Palanca A)
Gráfico sugerido: treemap / barras apiladas por categoría.

In [0]:
df_agg_funnel_categoria = spark.sql("""
    SELECT 
        macro_category,
        COUNT(1) AS units,
        SUM(CAST(has_cart OR has_purchase AS INT)) AS reached_cart,
        SUM(CAST(has_purchase AS INT)) AS purchased,
        ROUND(SUM(CASE WHEN has_purchase = TRUE THEN price ELSE 0 END), 0) AS revenue,
        ROUND(SUM(CASE WHEN has_cart = TRUE AND has_purchase = FALSE THEN price ELSE 0 END), 0) AS revenue_en_juego,
        ROUND((SUM(CAST(has_cart OR has_purchase AS INT)) / COUNT(1)) * 100, 2) AS cart_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / COUNT(1)) * 100, 2) AS conv_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT))) * 100, 2) AS cierre_pct,
        ROUND(100 - ((SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT))) * 100), 2) AS abandono_pct,
        ROUND(SUM(CASE WHEN has_purchase = TRUE THEN price ELSE 0 END) / SUM(CAST(has_purchase AS INT)), 2) AS ticket_medio
    FROM vw_units
    WHERE macro_category IS NOT NULL AND macro_category != 'Unknown'
    GROUP BY macro_category
    ORDER BY conv_rate DESC
""")

fc = df_agg_funnel_categoria.toPandas().set_index("macro_category")
fc = fc[["units", "reached_cart", "purchased", "revenue", "revenue_en_juego", "cart_rate", "conv_rate", "cierre_pct", "abandono_pct", "ticket_medio"]]
to_csv_sql(fc, "agg_funnel_categoria")
display(fc)

## 2. `agg_revenue_en_juego` — ¿cuánto $ hay en carritos abandonados? (Palanca A)
Gráfico sugerido: treemap (área = revenue en juego).

In [0]:
df_agg_revenue = spark.sql("""
    SELECT 
        macro_category,
        COUNT(1) AS carritos_abandonados,
        ROUND(SUM(price), 0) AS revenue_en_juego,
        ROUND(SUM(price) / COUNT(1), 2) AS ticket_medio
    FROM vw_units
    WHERE has_cart = TRUE AND has_purchase = FALSE
      AND macro_category IS NOT NULL AND macro_category != 'Unknown'
    GROUP BY macro_category
    ORDER BY revenue_en_juego DESC
""")

ab = df_agg_revenue.toPandas().set_index("macro_category")
ab = ab[["carritos_abandonados", "revenue_en_juego", "ticket_medio"]]
to_csv_sql(ab, "agg_revenue_en_juego")
display(ab)

## 3. `agg_marca_electronics` — ¿qué marcas concentran el premio? (Palanca A, drill)
Gráfico sugerido: barras (Samsung/Apple al frente).

In [0]:
# 3. agg_marca_electronics
df_agg_marca = spark.sql("""
    SELECT 
        brand,
        COUNT(1) AS carritos,
        SUM(CAST(has_purchase AS INT)) AS comprados,
        ROUND(percentile_approx(price, 0.5), 2) AS ticket
    FROM vw_units
    WHERE macro_category = 'electronics'
      AND (has_cart = TRUE OR has_purchase = TRUE)
      AND brand IS NOT NULL AND brand != 'Unknown'
    GROUP BY brand
""")

# Mantenemos EXACTAMENTE el mismo post-procesamiento en Pandas de Yeison
me = df_agg_marca.toPandas().set_index("brand")
me["abandonados"]  = me["carritos"] - me["comprados"]
me["abandono_pct"] = (me["abandonados"] / me["carritos"] * 100).round(1)
me = me[me["carritos"] >= 100].sort_values("abandonados", ascending=False)

to_csv_sql(me, "agg_marca_electronics")
display(me.head(15))

## 4. `agg_segmentos_comprador` — ¿qué segmento concentra el revenue? (Palanca B)
Gráfico sugerido: combo doble eje (% compradores vs % revenue). Ocasión = sesión distinta con compra.

In [0]:
df_agg_segmentos = spark.sql("""
    WITH user_stats AS (
        SELECT 
            user_id,
            COUNT(DISTINCT user_session) AS ocasiones,
            SUM(price) AS revenue,
            CASE WHEN COUNT(DISTINCT user_session) >= 2 THEN 'recurrente' ELSE 'one-time' END AS segmento
        FROM vw_silver
        WHERE event_type = 'purchase'
        GROUP BY user_id
    ),
    segment_totals AS (
        SELECT 
            segmento,
            COUNT(1) AS n_compradores,
            ROUND(SUM(revenue), 0) AS revenue,
            ROUND(AVG(revenue), 2) AS ticket_promedio
        FROM user_stats
        GROUP BY segmento
    )
    SELECT 
        segmento,
        n_compradores,
        revenue,
        ticket_promedio,
        ROUND((n_compradores / SUM(n_compradores) OVER ()) * 100, 1) AS pct_compradores,
        ROUND((revenue / SUM(revenue) OVER ()) * 100, 1) AS pct_revenue
    FROM segment_totals
    ORDER BY segmento
""")

seg = df_agg_segmentos.toPandas().set_index("segmento").sort_index()
seg = seg[["n_compradores", "revenue", "ticket_promedio", "pct_compradores", "pct_revenue"]]
to_csv_sql(seg, "agg_segmentos_comprador")
display(seg)

## 5. `agg_metricas_diarias` — evolución temporal (con banderas de calidad)
**Conserva TODOS los días** (incl. 14–17 nov) con `is_black_friday` y `ventana_corrupta` para que Kelly
cuente la historia de calidad de datos (anotación). Gráfico sugerido: líneas / áreas.

In [0]:
df_agg_diarias = spark.sql("""
    SELECT 
        CAST(date AS STRING) AS date,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
        COALESCE(ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 0), 0.0) AS revenue,
        ROUND((SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) / 
               SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END)) * 100, 3) AS conv_x100,
        CASE WHEN CAST(date AS STRING) = '2019-11-29' THEN 1 ELSE 0 END AS is_black_friday,
        CASE WHEN CAST(date AS STRING) IN ('2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17') THEN 1 ELSE 0 END AS ventana_corrupta
    FROM vw_silver_full
    GROUP BY date
    ORDER BY date
""")

daily = df_agg_diarias.toPandas().set_index("date")

# === EL FIX ARQUITECTÓNICO ===
# Replicamos el comportamiento del pivot de PySpark forzando el formato float
daily["purchases"] = daily["purchases"].astype(float)

daily = daily[["views", "carts", "purchases", "revenue", "conv_x100", "is_black_friday", "ventana_corrupta"]]
to_csv_sql(daily, "agg_metricas_diarias")
display(daily.tail(20))

## 6. `agg_tipologia_visitante` — browser / intender / buyer (nivel sesión)
Gráfico sugerido: barras (reparto y conversión). buyer = compra; intender = carrito sin compra; browser = resto.

In [0]:
df_agg_tipologia = spark.sql("""
    WITH session_types AS (
        SELECT 
            user_session,
            CASE 
                WHEN MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) = 1 THEN 'buyer'
                WHEN MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) = 1 THEN 'intender'
                ELSE 'browser'
            END AS tipo
        FROM vw_silver
        GROUP BY user_session
    )
    SELECT 
        tipo,
        COUNT(1) AS n_sesiones,
        ROUND((COUNT(1) / SUM(COUNT(1)) OVER ()) * 100, 2) AS pct
    FROM session_types
    GROUP BY tipo
""")

tip = df_agg_tipologia.toPandas().set_index("tipo").reindex(["browser", "intender", "buyer"])
tip = tip[["n_sesiones", "pct"]]
to_csv_sql(tip, "agg_tipologia_visitante")
display(tip)

In [0]:
# 7. agg_funnel_global -- funnel GLOBAL incl. 'Unknown' (tarjeta KPI del titular).
#    AUDITORIA 5-jun: los CSV por-categoria (1 y 2) EXCLUYEN 'Unknown' (~32% de unidades) -> al sumarlos
#    NO se reproduce el titular (cart 3.93 / conv 2.24 / abandono 43.1, 994k carritos, $283.6M). Esta
#    tabla SI cuadra con el funnel global del EDA (eda_ecommerce.ipynb 4.1): 58.6M unidades incl. Unknown.

df_agg_global = spark.sql("""
    SELECT 
        'GLOBAL (incl. Unknown)' AS scope,
        COUNT(1) AS units,
        SUM(CAST(has_cart OR has_purchase AS INT)) AS reached_cart,
        SUM(CAST(has_purchase AS INT)) AS purchased,
        ROUND((SUM(CAST(has_cart OR has_purchase AS INT)) / COUNT(1)) * 100, 2) AS cart_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / COUNT(1)) * 100, 2) AS conv_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT))) * 100, 2) AS cierre_pct,
        ROUND((1 - (SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT)))) * 100, 2) AS abandono_pct,
        SUM(CASE WHEN has_cart = TRUE AND has_purchase = FALSE THEN 1 ELSE 0 END) AS carritos_abandonados,
        ROUND(SUM(CASE WHEN has_cart = TRUE AND has_purchase = FALSE THEN price ELSE 0 END), 0) AS revenue_en_juego
    FROM vw_units
""")

fg = df_agg_global.toPandas().set_index("scope")
fg = fg[["units", "reached_cart", "purchased", "cart_rate", "conv_rate", "cierre_pct", "abandono_pct", "carritos_abandonados", "revenue_en_juego"]]
to_csv_sql(fg, "agg_funnel_global")
display(fg)

## agg_funnel_embudo

In [0]:
df_agg_embudo = spark.sql("""
    SELECT 
        COUNT(1) AS vistas,
        SUM(CAST(has_cart OR has_purchase AS INT)) AS en_carrito,
        SUM(CAST(has_purchase AS INT)) AS compra
    FROM vw_units
""")

# Extraemos la única fila como un diccionario de Pandas
e = df_agg_embudo.toPandas().iloc[0]

emb = pd.DataFrame([
    {"etapa": "1. Vistas",     "n": e["vistas"]},
    {"etapa": "2. En carrito", "n": e["en_carrito"]},
    {"etapa": "3. Compra",     "n": e["compra"]},
])

top = emb["n"].iloc[0]
emb["pct_del_total"]        = (emb["n"] / top * 100).round(2)
emb["pct_paso_anterior"]    = (emb["n"] / emb["n"].shift(1) * 100).round(2)
emb["perdidos_vs_anterior"] = (emb["n"].shift(1) - emb["n"]).fillna(0).astype("int64")

to_csv_sql(emb.set_index("etapa"), "agg_funnel_embudo")
display(emb)

## agg_hora_dow

In [0]:
# Aseguramos que la tabla gold esté registrada
gold.createOrReplaceTempView("vw_gold")

df_agg_hora = spark.sql("""
    SELECT 
        day_of_week,
        CASE 
            WHEN day_of_week = 1 THEN '1-Dom'
            WHEN day_of_week = 2 THEN '2-Lun'
            WHEN day_of_week = 3 THEN '3-Mar'
            WHEN day_of_week = 4 THEN '4-Mie'
            WHEN day_of_week = 5 THEN '5-Jue'
            WHEN day_of_week = 6 THEN '6-Vie'
            WHEN day_of_week = 7 THEN '7-Sab'
        END AS day_name,
        session_hour,
        MAX(CAST(is_weekend AS INT)) AS is_weekend,
        COUNT(1) AS n_sesiones,
        SUM(CAST(target_purchase AS INT)) AS n_compras,
        ROUND((SUM(CAST(target_purchase AS INT)) / COUNT(1)) * 100, 3) AS conv_x100
    FROM vw_gold
    GROUP BY day_of_week, session_hour
    ORDER BY day_of_week, session_hour
""")

hd = df_agg_hora.toPandas().set_index(["day_of_week", "session_hour"])
to_csv_sql(hd, "agg_hora_dow")
display(hd.head(12))

## agg_electronics_marca_diaria

In [0]:
df_agg_elec_diaria = spark.sql("""
    WITH TopBrands AS (
        SELECT brand
        FROM vw_silver_full
        WHERE macro_category = 'electronics' 
          AND brand IS NOT NULL AND brand != 'Unknown'
          AND event_type = 'cart'
        GROUP BY brand
        ORDER BY COUNT(1) DESC
        LIMIT 12
    ),
    Aggregates AS (
        SELECT 
            CAST(date AS STRING) AS date,
            brand,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
            COALESCE(ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 0), 0.0) AS revenue
        FROM vw_silver_full
        WHERE macro_category = 'electronics' 
          AND brand IN (SELECT brand FROM TopBrands)
        GROUP BY date, brand
    )
    SELECT 
        date,
        brand,
        views,
        carts,
        purchases,
        revenue,
        CASE WHEN date = '2019-11-29' THEN 1 ELSE 0 END AS is_black_friday,
        CASE WHEN date IN ('2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17') THEN 1 ELSE 0 END AS ventana_corrupta
    FROM Aggregates
    ORDER BY date, brand
""")

emb_b = df_agg_elec_diaria.toPandas().set_index(["date", "brand"])

# === EL FIX ARQUITECTÓNICO (Replicando el artefacto del pivot) ===
#emb_b["views"] = emb_b["views"].astype(float)
emb_b["carts"] = emb_b["carts"].astype(float)
emb_b["purchases"] = emb_b["purchases"].astype(float)

# Opcional: Para imprimir las marcas tal como hacía Yeison
top_brands_list = emb_b.index.get_level_values('brand').unique().tolist()
print("Top marcas electronics:", top_brands_list)

to_csv_sql(emb_b, "agg_electronics_marca_diaria")
display(emb_b.tail(12))

## agg_metricas_dia_hora

In [0]:
df_agg_dia_hora = spark.sql("""
    SELECT 
        CAST(date AS STRING) AS date,
        hour,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
        COALESCE(ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 0), 0.0) AS revenue,
        CASE WHEN CAST(date AS STRING) = '2019-11-29' THEN 1 ELSE 0 END AS is_black_friday,
        CASE WHEN CAST(date AS STRING) IN ('2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17') THEN 1 ELSE 0 END AS ventana_corrupta
    FROM vw_silver_full
    GROUP BY date, hour
    ORDER BY date, hour
""")

dh = df_agg_dia_hora.toPandas().set_index(["date", "hour"])
#dh["views"] = dh["views"].astype(float)
dh["carts"] = dh["carts"].astype(float)
dh["purchases"] = dh["purchases"].astype(float)
to_csv_sql(dh, "agg_metricas_dia_hora")
display(dh.tail(12))

In [0]:
# NUEVO (Kelly): agg_metricas_diarias_categoria
df_agg_diarias_cat = spark.sql("""
    SELECT 
        CAST(date AS STRING) AS date,
        macro_category,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
        COALESCE(ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 0), 0.0) AS revenue,
        CASE WHEN CAST(date AS STRING) = '2019-11-29' THEN 1 ELSE 0 END AS is_black_friday,
        CASE WHEN CAST(date AS STRING) IN ('2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17') THEN 1 ELSE 0 END AS ventana_corrupta
    FROM vw_silver_full
    GROUP BY date, macro_category
    ORDER BY date, macro_category
""")

dcat = df_agg_diarias_cat.toPandas().set_index(["date", "macro_category"])

# Reordenamos las columnas por si acaso, para asegurar que empaten con el PySpark
dcat = dcat[["views", "carts", "purchases", "revenue", "is_black_friday", "ventana_corrupta"]]

#dcat["views"] = dcat["views"].astype(float)
dcat["carts"] = dcat["carts"].astype(float)
dcat["purchases"] = dcat["purchases"].astype(float)

to_csv_sql(dcat, "agg_metricas_diarias_categoria")
display(dcat.tail(15))

## Cierre
Lista los CSV y recuerda el siguiente paso (commit a `reports/data/`).
El bloque del final borra el Delta temporal del EDA (`_tmp_eda_units`) — **descoméntalo cuando ya no lo uses** (Block 5).

In [0]:
print("CSVs en", BI_OUT_SQL, ":")
for f in sorted(os.listdir(BI_OUT_SQL)):
    print("  -", f)
print("\nSiguiente: descargar estos CSV y commitearlos en reports/data/ (los lee Power BI).")

# Limpieza del Delta temporal propio de este notebook (UNIT):
dbutils.fs.rm("/Volumes/workspace/default/e_commerce/gold/_tmp_bi_units", recurse=True)
print("borrado _tmp_bi_units")

# --- Block 5 (limpieza del Delta temporal del EDA) -- ejecutar cuando ya no se use ---
# dbutils.fs.rm("/Volumes/workspace/default/e_commerce/gold/_tmp_eda_units", recurse=True)

In [0]:
import pandas as pd
import os

# Rutas de las carpetas donde están guardados los resultados
BI_OUT_PYSPARK = "/Volumes/workspace/default/e_commerce/gold/bi_export"
BI_OUT_SQL = "/Volumes/workspace/default/e_commerce/gold/bi_export_SQL"

# Lista completa de los 12 archivos CSV generados
archivos = [
    "agg_funnel_categoria.csv",
    "agg_revenue_en_juego.csv",
    "agg_marca_electronics.csv",
    "agg_segmentos_comprador.csv",
    "agg_metricas_diarias.csv",
    "agg_tipologia_visitante.csv",
    "agg_funnel_global.csv",
    "agg_funnel_embudo.csv",
    "agg_hora_dow.csv",
    "agg_electronics_marca_diaria.csv",
    "agg_metricas_dia_hora.csv",
    "agg_metricas_diarias_categoria.csv"
]

print("🔍 Iniciando auditoría automatizada de los 12 CSVs...\n")

todos_iguales = True
archivos_procesados = 0

for archivo in archivos:
    ruta_pyspark = f"{BI_OUT_PYSPARK}/{archivo}"
    ruta_sql = f"{BI_OUT_SQL}/{archivo}"
    
    if os.path.exists(ruta_pyspark) and os.path.exists(ruta_sql):
        # Leemos ambos CSVs
        df_pyspark = pd.read_csv(ruta_pyspark)
        df_sql = pd.read_csv(ruta_sql)
        
        try:
            # assert_frame_equal verifica que valores, tipos de datos y columnas sean idénticos
            pd.testing.assert_frame_equal(df_pyspark, df_sql, check_exact=False, rtol=1e-3)
            print(f"✅ PASS: {archivo}")
            archivos_procesados += 1
        except AssertionError as e:
            print(f"❌ FAIL: {archivo} presenta diferencias.")
            print(e)
            todos_iguales = False
    else:
        print(f"⚠️ WARN: Falta el archivo {archivo} en alguna de las dos carpetas.")
        todos_iguales = False

print("-" * 50)
if todos_iguales and archivos_procesados == 12:
    print(f"🏆 AUDITORÍA 100% EXITOSA: Los {archivos_procesados}/12 archivos son matemáticamente perfectos.")
    print("La refactorización a Spark SQL está lista para producción.")
else:
    print("⚠️ Revisa los logs arriba para corregir las diferencias o generar los archivos faltantes.")